In [ ]:
from langchain_community.document_loaders import TextLoader

loader=TextLoader('speech.txt')
loader


In [ ]:
docs=loader.load()
docs

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
final_documents=text_splitter.split_documents(docs)
final_documents

In [ ]:
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores.faiss import FAISS
embeddings= OllamaEmbeddings(model="gemma2:2b")  ##by default it ues llama2
vectorstore=FAISS.from_documents(final_documents,embeddings)
vectorstore

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

In [ ]:
## Langsmith Tracking and tracing

os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")

In [ ]:
from langchain_ollama import ChatOllama
llm = ChatOllama(
    model="gemma2:2b",
    temperature=0,
    # other params...
)
print(llm)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt=ChatPromptTemplate.from_template(
    """
Answer the following question based only on the provided context:
<context>
{context}
</context>


"""
)

In [ ]:
retriever=vectorstore.as_retriever()

from langchain.chains.combine_documents import create_stuff_documents_chain
document_chain=create_stuff_documents_chain(llm,prompt)
document_chain

In [ ]:
from langchain.chains import create_retrieval_chain

retrieval_chain=create_retrieval_chain(retriever,document_chain)
retrieval_chain

In [ ]:
result=retrieval_chain.invoke({"input":"please provide information for the gentleman of congress"})

In [ ]:
result['answer']